In [24]:
!pip install torch -q

In [25]:
import torch
import numpy as np

print(f"PyTorch 版本: {torch.__version__}")
print(f"是否有 GPU: {torch.cuda.is_available()}")  # Mac 可能是 False，没关系，CPU 先跑着

PyTorch 版本: 2.11.0
是否有 GPU: False


In [26]:
# ========== NumPy vs PyTorch 对比 ==========

# 1. 创建
np_arr = np.array([[1.0, 2.0], [3.0, 4.0]])
torch_tensor = torch.tensor([[1.0, 2.0], [3.0, 4.0]])

print("NumPy:\n", np_arr)
print("\nPyTorch Tensor:\n", torch_tensor)

# 2. 运算（几乎一模一样）
print("\nNumPy 乘法:\n", np_arr * 2)
print("\nTensor 乘法:\n", torch_tensor * 2)

# 3. 互相转换（重要！以后经常要转）
torch_from_np = torch.from_numpy(np_arr)     # NumPy → Tensor
np_from_torch = torch_tensor.numpy()          # Tensor → NumPy（CPU上才能直接转）

print("\nNumPy → Tensor:\n", torch_from_np)
print("\nTensor → NumPy:\n", np_from_torch)

NumPy:
 [[1. 2.]
 [3. 4.]]

PyTorch Tensor:
 tensor([[1., 2.],
        [3., 4.]])

NumPy 乘法:
 [[2. 4.]
 [6. 8.]]

Tensor 乘法:
 tensor([[2., 4.],
        [6., 8.]])

NumPy → Tensor:
 tensor([[1., 2.],
        [3., 4.]], dtype=torch.float64)

Tensor → NumPy:
 [[1. 2.]
 [3. 4.]]


In [27]:
# ========== 自动求导：模拟 y = 2x + 1 ==========
# 目标：PyTorch 自动算出 dy/dx

x = torch.tensor(3.0, requires_grad=True)   # 告诉 PyTorch：我要对 x 求导
w = torch.tensor(2.0, requires_grad=True)   # 对 w 也求导
b = torch.tensor(1.0, requires_grad=True)

y = w * x + b       # 前向计算：y = 2*3 + 1 = 7

print(f"前向结果 y = {y.item()}")

# 反向传播：自动算梯度
y.backward()

print(f"dy/dw = {w.grad.item()}")   # 应该是 x 的值 = 3.0
print(f"dy/db = {b.grad.item()}")   # 应该是 1.0
print(f"dy/dx = {x.grad.item()}")   # 应该是 w 的值 = 2.0

前向结果 y = 7.0
dy/dw = 3.0
dy/db = 1.0
dy/dx = 2.0


In [28]:
# ========== 模拟线性回归的一步 ==========
# y_pred = w*x + b，然后算 MSE

x = torch.tensor([[1.0], [2.0], [3.0]])      # (3, 1)
y_true = torch.tensor([[3.0], [5.0], [7.0]]) # 真实值：y = 2x + 1

w = torch.tensor([[1.0]], requires_grad=True)  # 初始猜测 w=1
b = torch.tensor([[0.0]], requires_grad=True)  # 初始猜测 b=0

# 前向
y_pred = x @ w + b          # 矩阵乘法，和 NumPy 的 @ 一样
loss = torch.mean((y_pred - y_true) ** 2)   # MSE

print(f"预测:\n{y_pred.detach().numpy()}")
print(f"Loss: {loss.item():.4f}")

# 反向
loss.backward()

print(f"\n梯度 dL/dw: {w.grad.item():.4f}")   # 应该是负的，因为 w=1 比真实值 2 小
print(f"梯度 dL/db: {b.grad.item():.4f}")     # 应该是负的，因为 b=0 比真实值 1 小

预测:
[[1.]
 [2.]
 [3.]]
Loss: 9.6667

梯度 dL/dw: -13.3333
梯度 dL/db: -6.0000


In [29]:
# ========== 用 PyTorch 手动走一步梯度下降 ==========
# 不用优化器，纯手动，和你之前的 NumPy 代码对比

x = torch.tensor([[1.0], [2.0], [3.0]])
y_true = torch.tensor([[3.0], [5.0], [7.0]])

w = torch.tensor([[1.0]], requires_grad=True)
b = torch.tensor([[0.0]], requires_grad=True)

lr = 0.1

for epoch in range(10):
    # 前向
    y_pred = x @ w + b
    loss = torch.mean((y_pred - y_true) ** 2)
    
    # 反向（关键：每次反向前必须清零梯度，否则 PyTorch 会累加）
    w.grad = None   # 或者 w.zero_grad()
    b.grad = None
    loss.backward()
    
    # 手动更新（和 NumPy 一模一样）
    with torch.no_grad():       # 告诉 PyTorch：这步不算导数
        w -= lr * w.grad
        b -= lr * b.grad
    
    if epoch % 2 == 0:
        print(f"Epoch {epoch}: w={w.item():.4f}, b={b.item():.4f}, loss={loss.item():.4f}")



Epoch 0: w=2.3333, b=0.6000, loss=9.6667
Epoch 2: w=2.1935, b=0.5644, loss=0.0300
Epoch 4: w=2.1828, b=0.5845, loss=0.0260
Epoch 6: w=2.1741, b=0.6042, loss=0.0236
Epoch 8: w=2.1658, b=0.6230, loss=0.0214
